# Example 12.6.6 / 1: Compute VaR and CVaR of five losses

Companion notebook for **12.6.6 Risk Measures and CVaR** in the AMPL lab Documentation, section "Compute VaR and CVaR of five losses". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Compute VaR and CVaR of five losses

Five equally likely losses `10, 20, 30, 40, 100` (thousand dollars) have mean `40`. At
$\alpha = 0.7$ the lower VaR is `40`, because $\Pr\{L \le 30\} = 0.6 < 0.7$ and
$\Pr\{L \le 40\} = 0.8$. The tail of mass 0.3 holds `100` with weight 0.2 and `40` with weight
0.1, so CVaR is `(0.2*100 + 0.1*40)/0.3 = 80`. The formula gives the same number:
`40 + (0.2*60)/0.3 = 80`.

The model below is the scenario-based linear formulation of CVaR: `excess[s]` is the positive part
of the loss above `eta`, enforced by `TailLoss`, and the objective is the formula. It minimizes
over `eta` for fixed losses; the run changes `alpha` and reads `eta` and the objective with `value()`.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set SCENARIOS;                          # loss scenarios
param probability {SCENARIOS} >= 0;     # scenario weight
param loss {SCENARIOS};                 # thousand dollars
param alpha > 0 < 1;                    # confidence level of the tail

var eta;                                # VaR candidate
var excess {SCENARIOS} >= 0;            # loss above eta

subject to TailLoss {s in SCENARIOS}:   # excess is at least the loss above eta
    excess[s] >= loss[s] - eta;

minimize CVaR:                          # Rockafellar-Uryasev CVaR at level alpha
    eta + sum {s in SCENARIOS} probability[s] * excess[s] / (1 - alpha);

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["SCENARIOS"] = ["s1", "s2", "s3", "s4", "s5"]
ampl.param["probability"] = {s: 0.2 for s in ["s1", "s2", "s3", "s4", "s5"]}
ampl.param["loss"] = {"s1": 10, "s2": 20, "s3": 30, "s4": 40, "s5": 100}

In [5]:
# run.py
for level in [0.6, 0.7, 0.8]:
    ampl.param["alpha"] = level
    ampl.solve(solver="highs", verbose=False)
    print(level, ampl.solve_result, round(ampl.var["eta"].value(), 6),
          round(ampl.obj["CVaR"].value(), 6))

0.6 solved 40.0 70.0
0.7 solved 40.0 80.0
0.8 solved 40.0 100.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`; a `for` loop sets `alpha` with `let`:

In [6]:
%%writefile data.dat
set SCENARIOS := s1 s2 s3 s4 s5;
param probability default 0.2;
param loss := s1 10  s2 20  s3 30  s4 40  s5 100;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
for {level in {0.6, 0.7, 0.8}} {
    let alpha := level;
    solve;
    display alpha, eta, CVaR;
}

HiGHS 1.14.0: optimal solution; objective 70
5 simplex iterations
0 barrier iterations
alpha = 0.6
eta = 40
CVaR = 70

HiGHS 1.14.0: optimal solution; objective 80
0 simplex iterations
0 barrier iterations
alpha = 0.7
eta = 40
CVaR = 80

HiGHS 1.14.0: optimal solution; objective 100
0 simplex iterations
0 barrier iterations
alpha = 0.8
eta = 40
CVaR = 100

